# Data Importing and Preprocessing

### Preprocessing Philosophy

* Structural validity prioritized over predictive optimization  

* Deterministic relationships leveraged where possible  

* Imputation strategies selected to preserve empirical distributions  

* Domain knowledge integrated to resolve implausible configurations  

* Sequential reconstruction used to minimize propagation of uncertainty  

### Document Imports

In [587]:
import pandas as pd

### House Sales Dataset: Initial Structure and Descriptive Overview

* Dataset imported and a working copy created to preserve raw data integrity  

* Dimensions and variable types inspected to verify structural consistency prior to preprocessing  

* Numeric summary statistics generated to establish baseline distributional properties  

* Skewness included to identify asymmetric feature distributions requiring potential transformation or robust handling  

* Non-analytic identifiers (`id`, `date`) excluded from numeric summaries

In [588]:
print("Loading dataset and creating working copy...")

df = pd.read_csv("house_sales.csv")
clean_df = df.copy()

print("\nDataset dimensions (rows, columns):")
display(df.shape)

print("\nColumn data types and non-null counts:")
df.info()

print("\nDescriptive statistics for numeric variables "
      "(including skewness):")

df_num = df.drop(columns=['id', 'date'])

desc = df_num.describe()
desc.loc['skew'] = df_num.skew()

display(desc.round(2).T)

Loading dataset and creating working copy...

Dataset dimensions (rows, columns):


(21613, 21)


Column data types and non-null counts:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21613 entries, 0 to 21612
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             21613 non-null  int64  
 1   date           21613 non-null  object 
 2   price          21613 non-null  float64
 3   bedrooms       20479 non-null  float64
 4   bathrooms      20545 non-null  float64
 5   sqft_living    20503 non-null  float64
 6   sqft_lot       20569 non-null  float64
 7   floors         21613 non-null  float64
 8   waterfront     21613 non-null  int64  
 9   view           21613 non-null  int64  
 10  condition      21613 non-null  int64  
 11  grade          21613 non-null  int64  
 12  sqft_above     21613 non-null  int64  
 13  sqft_basement  21613 non-null  int64  
 14  yr_built       21613 non-null  int64  
 15  yr_renovated   21613 non-null  int64  
 16  zipcode        21613 non-null  int64  
 17  lat       

,count,mean,std,min,25%,50%,75%,max,skew
price,21613.0,540088.14,367127.20,75000.00,321950.00,450000.00,645000.00,7700000.00,4.02
bedrooms,20479.0,3.37,0.93,0.00,3.00,3.00,4.00,33.00,2.02
bathrooms,20545.0,2.11,0.77,0.00,1.50,2.25,2.50,8.00,0.50
sqft_living,20503.0,2081.07,915.04,290.00,1430.00,1920.00,2550.00,12050.00,1.39
sqft_lot,20569.0,15179.82,41486.17,520.00,5040.00,7620.00,10708.00,1651359.00,12.88
floors,21613.0,1.49,0.54,1.00,1.00,1.50,2.00,3.50,0.62
waterfront,21613.0,0.01,0.09,0.00,0.00,0.00,0.00,1.00,11.39
view,21613.0,0.23,0.77,0.00,0.00,0.00,0.00,4.00,3.40
condition,21613.0,3.41,0.65,1.00,3.00,3.00,4.00,5.00,1.03
grade,21613.0,7.66,1.18,1.00,7.00,7.00,8.00,13.00,0.77


### Missing Value Inspection

* Missing value counts and percentages computed to quantify the extent of incomplete observations  

* Variables with zero missingness excluded to focus attention on structurally relevant gaps  

* Co-occurrence of missingness examined across key housing attributes to assess potential dependency in missing mechanisms  

* Results used to guide statistically defensible imputation strategies in subsequent preprocessing steps  

In [589]:
def missing_summary(df):
    """
    Generate summary of missing values.

    Returns count and percentage of missing observations
    for variables with at least one missing entry.
    """
    na_counts = df.isna().sum()
    na_pct = (df.isna().mean() * 100).round(2)

    summary = pd.DataFrame({
        'Missing Count': na_counts,
        'Missing (%)': na_pct
    })

    return summary[summary['Missing Count'] > 0]

print("Evaluating extent of missing values across variables...")

# Display missing value summary
display(missing_summary(df))

print(
    "\nAssessing co-occurrence patterns in missingness "
    "among key structural variables:"
)

# Evaluate co-occurrence of missingness across key variables
na_corr = df[['bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot']].isna().corr()
display(na_corr.round(3))

Evaluating extent of missing values across variables...


,Missing Count,Missing (%)
bedrooms,1134,5.25
bathrooms,1068,4.94
sqft_living,1110,5.14
sqft_lot,1044,4.83



Assessing co-occurrence patterns in missingness among key structural variables:


,bedrooms,bathrooms,sqft_living,sqft_lot
bedrooms,1.000,0.003,0.020,0.009
bathrooms,0.003,1.000,0.002,0.014
sqft_living,0.020,0.002,1.000,-0.006
sqft_lot,0.009,0.014,-0.006,1.000


### Duplicate Record Validation

* Fully duplicated observations evaluated to confirm absence of data-entry replication  

* Recurrence of property identifiers examined to detect multiple recorded transactions  

* Distinction maintained between row-level duplication and legitimate entity-level repeat sales  

* Transactional frequency validated against expected dynamics of residential housing markets  

In [590]:
# Check for true duplicate rows and repeat property transactions
true_duplicates = df.duplicated().sum()

id_duplicates = (
    df['id']
    .value_counts()
    .gt(1)
    .sum()
)

display(
    f"There are {true_duplicates} true duplicate rows in the data."
)

display(
    f"There are {id_duplicates} houses that sold multiple times."
)

'There are 0 true duplicate rows in the data.'

'There are 176 houses that sold multiple times.'

### Date Variable Validation and Conversion

* Timestamp suffix inspected to verify absence of within-day temporal
  variation  

* Substring checks performed to confirm valid month and day ranges under
  YYYYMMDD formatting  

* String-based date variable converted to datetime to enable temporal
  analysis and ordering  

* Minimum and maximum transaction dates computed to establish dataset
  coverage window  

In [591]:
# Inspect whether timestamp component varies
print(
    "Inspecting timestamp component to confirm absence of "
    "meaningful intra-day variation:"
)

print(
    df['date']
    .str[-6:]
    .value_counts()
)

# Validate date structure (month and day ranges)
print(
    "\nValidating structural integrity of date formatting "
    "(month and day ranges):"
)

print(
    df['date']
    .str[4:6]
    .astype(int)
    .describe()
)  # month

print(
    df['date']
    .str[6:8]
    .astype(int)
    .describe()
)  # day

# Convert to datetime format
df['date'] = pd.to_datetime(
    df['date'],
    format='%Y%m%dT%H%M%S'
)

print(
    "\nValidating structural integrity of date formatting "
    "(month and day ranges):"
)

display(df['date'].head())

# Determine temporal coverage of dataset
min_date = df['date'].min()
max_date = df['date'].max()

duration = max_date.date() - min_date.date()

print(
    f"Dataset spans from {min_date.date()} to {max_date.date()} "
    f"({duration} total duration)."
)

Inspecting timestamp component to confirm absence of meaningful intra-day variation:
date
000000    21613
Name: count, dtype: int64

Validating structural integrity of date formatting (month and day ranges):
count    21613.000000
mean         6.574423
std          3.115308
min          1.000000
25%          4.000000
50%          6.000000
75%          9.000000
max         12.000000
Name: date, dtype: float64
count    21613.000000
mean        15.688197
std          8.635063
min          1.000000
25%          8.000000
50%         16.000000
75%         23.000000
max         31.000000
Name: date, dtype: float64

Validating structural integrity of date formatting (month and day ranges):


0   2014-10-13
1   2014-12-09
2   2015-02-25
3   2014-12-09
4   2015-02-18
Name: date, dtype: datetime64[ns]

Dataset spans from 2014-05-02 to 2015-05-27 (390 days, 0:00:00 total duration).


### Deterministic Reconstruction of Living Area

* Arithmetic consistency of living area evaluated using the identity
  `sqft_living = sqft_above + sqft_basement`  

* Missing `sqft_living` values reconstructed only where component values
  allowed deterministic recovery  

* Pre- and post-reconstruction diagnostics compared to verify identity
  preservation and quantify imputation impact  

* Remaining dataset-level missingness reviewed after reconstruction to
  confirm downstream preprocessing needs  

In [592]:
# Define function to validate arithmetic identity and reconstruct
# sqft_living
def validate_and_reconstruct_living(df):
    """
    Validate identity: sqft_living = sqft_above + sqft_basement.
    Reconstruct missing sqft_living values deterministically.
    Return before/after diagnostics.
    """
    diff_before = (
        df['sqft_living']
        - (df['sqft_above'] + df['sqft_basement'])
    ).describe()

    missing_before = df['sqft_living'].isna().sum()

    # Deterministic reconstruction
    df.loc[df['sqft_living'].isna(), 'sqft_living'] = (
        df['sqft_above'] + df['sqft_basement']
    )

    diff_after = (
        df['sqft_living']
        - (df['sqft_above'] + df['sqft_basement'])
    ).describe()

    missing_after = df['sqft_living'].isna().sum()

    return diff_before, diff_after, missing_before, missing_after


# Run reconstruction pipeline
before, after, miss_before, miss_after = (
    validate_and_reconstruct_living(df)
)

print("Consistency check BEFORE reconstruction")
display(before.to_frame(name='Difference'))

print("Consistency check AFTER reconstruction")
display(after.to_frame(name='Difference'))

print(f"Missing sqft_living before: {miss_before}")
print(f"Missing sqft_living after: {miss_after}")

print("\nRemaining missing values in dataset")
display(missing_summary(df))

Consistency check BEFORE reconstruction


,Difference
count,20503.0
mean,0.0
std,0.0
min,0.0
25%,0.0
50%,0.0
75%,0.0
max,0.0


Consistency check AFTER reconstruction


,Difference
count,21613.0
mean,0.0
std,0.0
min,0.0
25%,0.0
50%,0.0
75%,0.0
max,0.0


Missing sqft_living before: 1110
Missing sqft_living after: 0

Remaining missing values in dataset


,Missing Count,Missing (%)
bedrooms,1134,5.25
bathrooms,1068,4.94
sqft_lot,1044,4.83


### Missingness Mechanism Diagnostics

* Binary indicators constructed to represent the presence of missing
  observations in key structural variables  

* Associations between missingness and selected economic and structural
  predictors evaluated using correlation analysis  

* Observed patterns assessed for consistency with approximately random
  or weakly structured missingness behavior  

* Results used to guide selection of statistically defensible
  imputation strategies  

In [593]:
# Create missingness indicators
df['bed_missing'] = df['bedrooms'].isna().astype(int)
df['bath_missing'] = df['bathrooms'].isna().astype(int)
df['lot_missing'] = df['sqft_lot'].isna().astype(int)

# Evaluate correlation between missingness and key predictors
missing_corr = (
    df[
        [
            'bed_missing',
            'bath_missing',
            'lot_missing',
            'price',
            'sqft_living',
            'grade',
            'floors',
            'lat',
            'long',
        ]
    ]
    .corr()
    .loc[
        ['bed_missing', 'bath_missing', 'lot_missing'],
        [
            'price',
            'sqft_living',
            'grade',
            'floors',
            'lat',
            'long',
        ],
    ]
)

print(
    "Correlation between missingness indicators and "
    "key structural and economic variables:"
)

display(missing_corr.round(3))

Correlation between missingness indicators and key structural and economic variables:


,price,sqft_living,grade,floors,lat,long
bed_missing,-0.004,-0.007,-0.001,0.003,-0.001,-0.000
bath_missing,-0.000,0.002,0.010,0.007,-0.002,0.013
lot_missing,-0.008,-0.003,-0.003,0.005,-0.001,0.001


### Logical Constraint Validation: Structural Housing Attributes

* Observations screened for implausible residential configurations
  violating basic housing constraints  

* Records with non-positive values for both bedrooms and bathrooms
  identified as structurally invalid  

* Such cases evaluated as potential data entry errors or
  non-residential parcels  

* Findings used to guide recoding or exclusion decisions in subsequent
  preprocessing steps  

In [594]:
# Identify observations with either bedrooms and bathrooms reported
# as zero or less
print(
    "Diagnostic inspection of observations with structurally "
    "invalid housing counts (bedrooms ≤ 0 or bathrooms ≤ 0):"
)

invalid_structures = df.loc[
    (df['bedrooms'] <= 0)
    | (df['bathrooms'] <= 0),
    [
        'price',
        'bedrooms',
        'bathrooms',
        'sqft_living',
        'sqft_lot',
        'floors',
        'grade',
        'yr_built',
        'zipcode'
    ],
]

print(f"Number of observations flagged: {len(invalid_structures)}")

display(invalid_structures)

Diagnostic inspection of observations with structurally invalid housing counts (bedrooms ≤ 0 or bathrooms ≤ 0):
Number of observations flagged: 15


,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,grade,yr_built,zipcode
875,1095000.0,0.0,0.00,3064.0,4764.0,3.5,7,1990,98102
1149,75000.0,1.0,0.00,670.0,43377.0,1.0,3,1966,98022
3119,380000.0,NaN,0.00,1470.0,979.0,3.0,8,2006,98133
3467,288000.0,0.0,NaN,1430.0,1650.0,3.0,7,1999,98125
4868,228000.0,0.0,1.00,390.0,5900.0,1.0,4,1953,98118
5832,280000.0,1.0,0.00,600.0,24501.0,1.0,3,1950,98045
6994,1295650.0,0.0,0.00,4810.0,28008.0,2.0,12,1990,98053
8477,339950.0,0.0,2.50,2290.0,8319.0,2.0,8,1985,98042
8484,240000.0,0.0,2.50,1810.0,5669.0,2.0,7,2003,98038
9773,355000.0,0.0,0.00,2460.0,8049.0,2.0,8,1990,98031


### Evaluation of Spatial Imputation Strategies for `sqft_lot`

* Lot size exhibits strong spatial heterogeneity and right-skew,
  motivating geographically informed imputation strategies  

* Although `sqft_lot15` provides hyper-local neighborhood context,
  it represents a derived spatial statistic rather than an intrinsic
  parcel-level attribute  

* Empirical comparison indicated that direct imputation using
  `sqft_lot15` increased between-zipcode variance, reflecting
  amplification of micro-spatial gradients rather than preservation
  of the underlying structural data-generating process  

* Global median and zipcode-conditional median imputation therefore
  evaluated as structurally consistent candidate approaches  

* Between-zipcode variance of median lot size compared before and
  after imputation to assess preservation of spatial structure  

In [595]:
# Baseline correlation check (contextual diagnostic)
print(
    "Correlation between sqft_lot and sqft_lot15 "
    "(complete cases):"
)

display(
    df[['sqft_lot', 'sqft_lot15']]
    .corr()
    .round(3)
)


# ---------------- Observed spatial variance ----------------
observed_var = (
    df
    .dropna(subset=['sqft_lot'])
    .groupby('zipcode')['sqft_lot']
    .median()
    .var()
)

# ---------------- Lot15 imputation diagnostic ----------------
sqft_lot_15imp = df['sqft_lot'].fillna(df['sqft_lot15'])

lot15_var = (
    pd.DataFrame(
        {
            'zipcode': df['zipcode'],
            'lot': sqft_lot_15imp,
        }
    )
    .groupby('zipcode')['lot']
    .median()
    .var()
)

# ---------------- Global median imputation ----------------
global_median = df['sqft_lot'].median()

sqft_lot_global = df['sqft_lot'].fillna(global_median)

global_var = (
    pd.DataFrame(
        {
            'zipcode': df['zipcode'],
            'lot': sqft_lot_global,
        }
    )
    .groupby('zipcode')['lot']
    .median()
    .var()
)


# ---------------- Zipcode median imputation ----------------
zip_median = (
    df
    .groupby('zipcode')['sqft_lot']
    .median()
)

sqft_lot_zip = df['sqft_lot'].fillna(
    df['zipcode'].map(zip_median)
)

zip_var = (
    pd.DataFrame(
        {
            'zipcode': df['zipcode'],
            'lot': sqft_lot_zip,
        }
    )
    .groupby('zipcode')['lot']
    .median()
    .var()
)


print("\nBetween-zipcode variance of median lot size:")
print(f"\nObserved: {observed_var:,.2f}")
print(f"Global median imputation: {global_var:,.2f}")
print(f"Zipcode median imputation: {zip_var:,.2f}")
print(f"Lot15 imputation: {lot15_var:,.2f}")

Correlation between sqft_lot and sqft_lot15 (complete cases):


,sqft_lot,sqft_lot15
sqft_lot,1.000,0.729
sqft_lot15,0.729,1.000



Between-zipcode variance of median lot size:

Observed: 61,120,480.97
Global median imputation: 54,302,396.90
Zipcode median imputation: 61,120,480.97
Lot15 imputation: 65,143,316.53


### Zipcode-Conditional Median Imputation for `sqft_lot`

* Global median imputation materially reduced between-zipcode variance,
  indicating attenuation of spatial heterogeneity  

* Zipcode-conditional median imputation preserved the observed variance
  structure, maintaining realistic geographic differentiation in parcel
  size  

* Given the strong location dependence of lot size in housing markets,
  zipcode-conditional median imputation adopted as the final strategy  

In [596]:
# Zipcode-conditional median lot size
lot_zip_median = df.groupby('zipcode')['sqft_lot'].median()

# Impute missing sqft_lot
df['sqft_lot'] = df['sqft_lot'].fillna(
    df['zipcode'].map(lot_zip_median)
)

print("Remaining missing sqft_lot:")
print(df['sqft_lot'].isna().sum())

Remaining missing sqft_lot:
0


### Bedroom Count Structural Validation

* Extremely high bedroom counts screened for structural plausibility
  relative to observed residential configurations  

* One observation with 33 bedrooms and modest living area identified
  as inconsistent with empirical housing distributions  

* Conditional comparison against three-bedroom living area supported
  interpretation as a data entry error  

* Value corrected prior to imputation to prevent distortion of ratio-
  based reconstruction procedures  

In [597]:
# Inspect extra large bedroom counts
print(
    "Observations with extra large bedroom counts "
    "(bedrooms ≥ 10):"
)

beds_above_ten = df.loc[
    df['bedrooms'] >= 10,
    [
        'bedrooms',
        'bathrooms',
        'sqft_living',
        'grade',
        'price',
        'floors',
    ],
].T

display(beds_above_ten)

# Compare against the distribution of 3-bedroom living area
print("Distribution of sqft_living for 3-bedroom homes:")

three_bed_stats = (
    df.loc[df['bedrooms'] == 3, 'sqft_living']
    .describe()
)

display(three_bed_stats)

# Correct clear data entry error
df.loc[df['bedrooms'] == 33, 'bedrooms'] = 3

Observations with extra large bedroom counts (bedrooms ≥ 10):


,13314,15161,15870,19254
bedrooms,10.00,10.0,33.00,10.0
bathrooms,5.25,2.0,1.75,3.0
sqft_living,4590.00,3610.0,1620.00,2920.0
grade,9.00,7.0,7.00,7.0
price,1148000.00,650000.0,640000.00,660000.0
floors,1.00,2.0,1.00,2.0


Distribution of sqft_living for 3-bedroom homes:


count    9286.000000
mean     1806.138273
std       622.920862
min       490.000000
25%      1370.000000
50%      1680.000000
75%      2110.000000
max      6400.000000
Name: sqft_living, dtype: float64

### Bedroom Count Harmonization

* Empirical bedroom distribution evaluated to identify structurally
  implausible residential configurations  

* Zero-bedroom observations assessed relative to the lower tail of
  valid one-bedroom homes  

* Such cases interpreted as measurement artifacts rather than
  meaningful housing typologies  

* Zero-bedroom values recoded to null to support statistically
  coherent downstream imputation  

In [598]:
# Bedroom distribution
print(
    "Reviewing empirical distribution of bedroom counts:"
)

display(
    df['bedrooms']
    .describe()
)

display(
    df['bedrooms']
    .value_counts()
)

# Inspect zero-bedroom observations separately
print(
    "\nDiagnostic inspection of observations with "
    "bedrooms < 1:"
)

beds_below_one = df.loc[
    df['bedrooms'] < 1,
    [
        'bedrooms',
        'bathrooms',
        'sqft_living',
        'grade',
        'price',
        'floors',
    ],
].T

print(f"Number of observations flagged: {beds_below_one.shape[1]}")

display(beds_below_one)

# Q1 threshold for 1-bedroom homes
q1_1bed = (
    df.loc[df['bedrooms'] == 1, 'sqft_living']
    .quantile(0.25)
)

# Count 1-bedroom homes below Q1 threshold
small_1beds = df.loc[
    (df['bedrooms'] == 1)
    & (df['sqft_living'] < q1_1bed)
]

print(f"Q1 threshold for 1-bedroom sqft: {q1_1bed}")

print(
    f"Number of 1-bedroom homes below Q1: "
    f"{len(small_1beds)}"
)

print(
    "\nSqft_living distribution for 1-bedroom homes "
    "below the first quartile threshold:"
)

display(
    small_1beds['sqft_living']
    .describe()
)

print("Bedroom missing after structural harmonization:")

print(
    df['bedrooms']
    .isna()
    .sum()
)

Reviewing empirical distribution of bedroom counts:


count    20479.000000
mean         3.371356
std          0.907394
min          0.000000
25%          3.000000
50%          3.000000
75%          4.000000
max         10.000000
Name: bedrooms, dtype: float64

bedrooms
3.0     9287
4.0     6519
2.0     2617
5.0     1539
6.0      263
1.0      189
7.0       34
8.0       12
0.0       11
9.0        5
10.0       3
Name: count, dtype: int64


Diagnostic inspection of observations with bedrooms < 1:
Number of observations flagged: 11


,875,3467,4868,6994,8477,8484,9773,9854,12653,14423,18379
bedrooms,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00
bathrooms,0.0,NaN,1.0,0.0,2.5,2.5,0.0,0.0,2.5,NaN,0.75
sqft_living,3064.0,1430.0,390.0,4810.0,2290.0,1810.0,2460.0,1470.0,1490.0,844.0,384.00
grade,7.0,7.0,4.0,12.0,8.0,7.0,8.0,7.0,7.0,7.0,4.00
price,1095000.0,288000.0,228000.0,1295650.0,339950.0,240000.0,355000.0,235000.0,320000.0,139950.0,265000.00
floors,3.5,3.0,1.0,2.0,2.0,2.0,2.0,2.0,2.0,1.0,1.00


Q1 threshold for 1-bedroom sqft: 650.0
Number of 1-bedroom homes below Q1: 47

Sqft_living distribution for 1-bedroom homes below the first quartile threshold:


count     47.000000
mean     541.914894
std       77.252784
min      370.000000
25%      490.000000
50%      560.000000
75%      610.000000
max      640.000000
Name: sqft_living, dtype: float64

Bedroom missing after structural harmonization:
1134


### Bedroom Imputation Strategy Comparison

* Structurally invalid bedroom counts recoded to missing prior to
  statistical reconstruction  

* Ratio-based imputation strategies evaluated using global,
  grade-conditioned, and bathroom-conditioned sqft-per-bedroom
  relationships  

* Distributional preservation assessed via normalized frequency
  comparison against observed bedroom counts  

* Methods additionally screened for implausible high-bedroom
  imputations to ensure structural realism  

* Strategy selection based on stability, interpretability, and
  consistency with empirical housing structure  

In [599]:
# Recode structurally invalid bedroom values first
df.loc[df['bedrooms'] < 1, 'bedrooms'] = pd.NA

df['bedrooms'] = df['bedrooms'].astype('Int64')

# Preserve original bedroom series
bed_original = df['bedrooms'].copy()

# Use observed, valid bedroom values only to estimate ratios
df_temp = df.loc[df['bedrooms'].notna()].copy()

df_temp['sqft_per_bedroom'] = (
    df_temp['sqft_living'] / df_temp['bedrooms']
)

# Mask for rows requiring bedroom imputation
mask_bed_missing = bed_original.isna()

# -----------------------------
# Global ratio method
# -----------------------------
global_ratio = df_temp['sqft_per_bedroom'].median()

bed_ratio_global = bed_original.fillna(
    (df['sqft_living'] / global_ratio).round()
)

# -----------------------------
# Grade-conditioned ratio method
# -----------------------------
grade_ratio = (
    df_temp
    .groupby('grade')['sqft_per_bedroom']
    .median()
)

bed_ratio_grade = bed_original.fillna(
    (
        df['sqft_living']
        / df['grade'].map(grade_ratio)
    ).round()
)

# -----------------------------
# Bathroom-conditioned ratio method
# -----------------------------
bath_ratio = (
    df_temp
    .groupby('bathrooms')['sqft_per_bedroom']
    .median()
)

bed_ratio_bath = bed_original.fillna(
    (
        df['sqft_living']
        / df['bathrooms'].map(bath_ratio)
    ).round()
)

# -----------------------------
# Compare normalized frequency distributions
# -----------------------------
dist_observed = (
    bed_original
    .dropna()
    .value_counts(normalize=True)
    .sort_index()
)

dist_global = (
    bed_ratio_global
    .value_counts(normalize=True)
    .sort_index()
)

dist_grade = (
    bed_ratio_grade
    .value_counts(normalize=True)
    .sort_index()
)

dist_bath = (
    bed_ratio_bath
    .value_counts(normalize=True)
    .sort_index()
)

imputation_comparison = (
    pd.concat(
        [
            dist_observed,
            dist_global,
            dist_grade,
            dist_bath,
        ],
        axis=1,
    )
    .fillna(0)
)

imputation_comparison.columns = [
    'Observed',
    'Global Ratio',
    'Grade Ratio',
    'Bathroom Ratio',
]

print("Bedroom imputation strategy distribution comparison:")

display(imputation_comparison)

# -----------------------------
# Flag implausible high-bedroom imputations
# -----------------------------
print("Global ratio imputations >= 10 bedrooms:")

display(
    df.loc[
        mask_bed_missing
        & (bed_ratio_global >= 10),
        ['sqft_living', 'grade', 'bathrooms'],
    ]
)

print("Grade ratio imputations >= 10 bedrooms:")

display(
    df.loc[
        mask_bed_missing
        & (bed_ratio_grade >= 10),
        ['sqft_living', 'grade', 'bathrooms'],
    ]
)

print("Bathroom ratio imputations >= 10 bedrooms:")

display(
    df.loc[
        mask_bed_missing
        & (bed_ratio_bath >= 10),
        ['sqft_living', 'grade', 'bathrooms'],
    ]
)

print(
    "Remaining missing bedroom values after "
    "global-conditioned imputation:"
)

print(bed_ratio_global.isna().sum())

print(
    "Remaining missing bedroom values after "
    "grade-conditioned imputation:"
)

print(bed_ratio_grade.isna().sum())

print(
    "Remaining missing bedroom values after "
    "bathroom-conditioned imputation:"
)

print(bed_ratio_bath.isna().sum())

Bedroom imputation strategy distribution comparison:


,Observed,Global Ratio,Grade Ratio,Bathroom Ratio
bedrooms,,,,
1,0.009234,0.010503,0.008976,0.009048
2,0.127858,0.132929,0.131408,0.130707
3,0.453733,0.445056,0.448408,0.45026
4,0.318497,0.313608,0.318342,0.316351
5,0.075191,0.078194,0.076624,0.077348
6,0.012849,0.015269,0.013418,0.01327
7,0.001661,0.002545,0.001758,0.001902
8,0.000586,0.001203,0.000648,0.000696
9,0.000244,0.00037,0.000231,0.000232


Global ratio imputations >= 10 bedrooms:


,sqft_living,grade,bathrooms
527,6050.0,11,5.00
9253,5584.0,11,4.25
11829,5490.0,12,3.50
12777,13540.0,12,8.00


Grade ratio imputations >= 10 bedrooms:


,sqft_living,grade,bathrooms
12777,13540.0,12,8.0


Bathroom ratio imputations >= 10 bedrooms:


,sqft_living,grade,bathrooms


Remaining missing bedroom values after global-conditioned imputation:
0
Remaining missing bedroom values after grade-conditioned imputation:
1
Remaining missing bedroom values after bathroom-conditioned imputation:
61


### Bedroom Imputation Using Bathroom-Conditioned Ratios

* Bedroom counts imputed using bathroom-conditioned sqft-per-bedroom
  ratios to leverage structural co-dependence between these features  

* This approach preserved the empirical bedroom distribution and
  reduced incidence of implausibly high-bedroom imputations  

* Residual missing bedroom values deferred pending completion of
  bathroom imputation to ensure sequential consistency  

In [600]:
print(
    "Imputing missing bedroom counts using the "
    "bathroom-conditioned sqft-per-bedroom ratio:"
)

missing_before = df['bedrooms'].isna().sum()

df['bedrooms'] = df['bedrooms'].fillna(
    (
        df['sqft_living']
        / df['bathrooms'].map(bath_ratio)
    )
    .round()
    .clip(lower=1)
)

missing_after = df['bedrooms'].isna().sum()

print(f"Missing bedrooms before imputation: {missing_before}")
print(f"Missing bedrooms after imputation: {missing_after}")

Imputing missing bedroom counts using the bathroom-conditioned sqft-per-bedroom ratio:
Missing bedrooms before imputation: 1145
Missing bedrooms after imputation: 61


### Bathroom Imputation Strategy

* Structurally invalid zero-bathroom observations recoded to missing
  prior to statistical reconstruction  

* Ratio-based imputation grounded in the empirical relationship
  between interior living area and bathroom count  

* Sqft-per-bath ratios estimated conditional on housing grade to
  reflect architectural and quality-tier heterogeneity  

* Imputed values quantized to 0.25 increments to preserve observed
  measurement resolution and structural realism  

* Grade-conditioned strategy selected over global ratio due to
  superior distributional preservation and reduced structural
  distortion  

In [601]:
# Recode structurally invalid bathroom values first
df.loc[df['bathrooms'] == 0, 'bathrooms'] = pd.NA

# ==========================================
# Preserve original bathroom series
# ==========================================
bath_original = df['bathrooms'].copy()

print("Observed bathroom distribution (complete cases only):")

display(
    bath_original
    .dropna()
    .describe()
)

print("\nObserved bathroom frequency distribution:")

display(
    bath_original
    .dropna()
    .value_counts(normalize=True)
    .sort_index()
)

# ==========================================
# Build sqft-per-bath ratio from observed data
# ==========================================
df_temp = df.loc[df['bathrooms'].notna()].copy()

df_temp['sqft_per_bath'] = (
    df_temp['sqft_living'] / df_temp['bathrooms']
)

mask_missing = bath_original.isna()

# ==========================================
# Global ratio method (quantized)
# ==========================================
global_ratio_bath = df_temp['sqft_per_bath'].median()

bath_ratio_global = bath_original.copy()

bath_ratio_global.loc[mask_missing] = (
    df.loc[mask_missing, 'sqft_living']
    / global_ratio_bath
)

bath_ratio_global.loc[mask_missing] = (
    (bath_ratio_global.loc[mask_missing] * 4)
    .round()
    / 4
).clip(lower=0.5)

# ==========================================
# Grade-conditioned ratio method (quantized)
# with global fallback for uncovered grades
# ==========================================
grade_ratio_bath = (
    df_temp
    .groupby('grade')['sqft_per_bath']
    .median()
)

bath_ratio_grade = bath_original.copy()

# First attempt: grade-conditioned fill
bath_ratio_grade.loc[mask_missing] = (
    df.loc[mask_missing, 'sqft_living']
    / df.loc[mask_missing, 'grade'].map(grade_ratio_bath)
)

# Fallback to global ratio if grade lookup missing
mask_still_missing = bath_ratio_grade.isna()

bath_ratio_grade.loc[mask_still_missing] = (
    df.loc[mask_still_missing, 'sqft_living']
    / global_ratio_bath
)

# Quantize to quarter-bath increments
bath_ratio_grade.loc[mask_missing] = (
    (bath_ratio_grade.loc[mask_missing] * 4)
    .round()
    / 4
).clip(lower=0.5)

# ==========================================
# Compare normalized frequency distributions
# ==========================================
dist_observed = (
    bath_original
    .dropna()
    .value_counts(normalize=True)
    .sort_index()
)

dist_global = (
    bath_ratio_global
    .value_counts(normalize=True)
    .sort_index()
)

dist_grade = (
    bath_ratio_grade
    .value_counts(normalize=True)
    .sort_index()
)

bath_imputation_comparison = (
    pd.concat(
        [
            dist_observed,
            dist_global,
            dist_grade,
        ],
        axis=1,
    )
    .fillna(0)
)

bath_imputation_comparison.columns = [
    'Observed',
    'Global Ratio',
    'Grade Ratio',
]

print("Bathroom imputation strategy distribution comparison:")

display(bath_imputation_comparison)

Observed bathroom distribution (complete cases only):


count    20537.00000
mean         2.11433
std          0.76793
min          0.50000
25%          1.75000
50%          2.25000
75%          2.50000
max          8.00000
Name: bathrooms, dtype: float64


Observed bathroom frequency distribution:


bathrooms
0.50    0.000195
0.75    0.003311
1.00    0.178556
1.25    0.000390
1.50    0.067342
1.75    0.141257
2.00    0.089010
2.25    0.094999
2.50    0.249063
2.75    0.054584
3.00    0.034718
3.25    0.027170
3.50    0.033598
3.75    0.007304
4.00    0.006233
4.25    0.003749
4.50    0.004577
4.75    0.001120
5.00    0.001023
5.25    0.000487
5.50    0.000487
5.75    0.000146
6.00    0.000243
6.25    0.000097
6.50    0.000097
6.75    0.000049
7.50    0.000049
7.75    0.000049
8.00    0.000097
Name: proportion, dtype: float64

Bathroom imputation strategy distribution comparison:


,Observed,Global Ratio,Grade Ratio
bathrooms,,,
0.50,0.000195,0.000370,0.000324
0.75,0.003311,0.004904,0.004581
1.00,0.178556,0.173229,0.172489
1.25,0.000390,0.005552,0.005043
1.50,0.067342,0.070097,0.070421
1.75,0.141257,0.139361,0.139314
2.00,0.089010,0.090779,0.090640
2.25,0.094999,0.094434,0.095914
2.50,0.249063,0.241244,0.241521


### Bathroom Imputation Using Grade-Conditioned Ratios

* Missing bathroom values reconstructed using grade-conditioned
  sqft-per-bath ratios to reflect quality-tier differences in housing
  design  

* Global ratio applied as fallback for observations lacking valid
  grade-conditioned estimates  

* Imputed values quantized to 0.25 increments to preserve observed
  measurement resolution and structural realism  

In [602]:
print(
    "Imputing missing bathroom counts using the "
    "grade-conditioned sqft-per-bath ratio:"
)

missing_before = df['bathrooms'].isna().sum()

print(f"Missing bathrooms before imputation: {missing_before}")

# Primary imputation: grade-conditioned ratio
df['bathrooms'] = df['bathrooms'].fillna(
    df['sqft_living']
    / df['grade'].map(grade_ratio_bath)
)

missing_after_grade = df['bathrooms'].isna().sum()

print(
    f"Missing bathrooms after grade-conditioned "
    f"imputation: {missing_after_grade}"
)

# Fallback: global ratio
df['bathrooms'] = df['bathrooms'].fillna(
    df['sqft_living'] / global_ratio_bath
)

missing_after_global = df['bathrooms'].isna().sum()

print(
    f"Missing bathrooms after global fallback: "
    f"{missing_after_global}"
)

# Quantization + structural constraint
df['bathrooms'] = (
    (df['bathrooms'] * 4)
    .round()
    / 4
).clip(lower=0.5)

print(
    "Minimum bathroom value after quantization:",
    df['bathrooms'].min()
)

print(
    "Total missing bathrooms after reconstruction:",
    df['bathrooms'].isna().sum()
)

Imputing missing bathroom counts using the grade-conditioned sqft-per-bath ratio:
Missing bathrooms before imputation: 1076
Missing bathrooms after grade-conditioned imputation: 1
Missing bathrooms after global fallback: 0
Minimum bathroom value after quantization: 0.5
Total missing bathrooms after reconstruction: 0


### Final Bedroom Imputation (Residual Missing Values)

* Residual missing bedroom values addressed using a staged fallback
  imputation strategy  

* Global sqft-per-bedroom ratio estimated from the original complete
  dataset to avoid circular reconstruction bias  

* Conditional alternatives evaluated but found not to materially
  improve distributional fidelity for remaining tail observations  

* Global ratio adopted as statistically stable final step to ensure
  full dataset completeness while minimizing additional variance
  distortion  

In [603]:
# ==========================================
# Build sqft-per-bedroom ratio from ORIGINAL data only
# ==========================================
df_temp_orig = clean_df.loc[
    clean_df['bedrooms'].notna()
].copy()

df_temp_orig['sqft_per_bedroom'] = (
    df_temp_orig['sqft_living']
    / df_temp_orig['bedrooms']
)

# Global ratio (true observed)
global_ratio_orig = df_temp_orig['sqft_per_bedroom'].median()

# Grade-conditioned ratio (true observed)
grade_ratio_orig = (
    df_temp_orig
    .groupby('grade')['sqft_per_bedroom']
    .median()
)

# ==========================================
# Candidate imputations for remaining NA bedrooms
# ==========================================
bed_global_candidate = df['bedrooms'].copy()
bed_grade_candidate = df['bedrooms'].copy()

mask_remaining = df['bedrooms'].isna()

# Global ratio imputation
bed_global_candidate.loc[mask_remaining] = (
    df.loc[mask_remaining, 'sqft_living']
    / global_ratio_orig
).round().clip(lower=1)

# Grade ratio imputation
bed_grade_candidate.loc[mask_remaining] = (
    df.loc[mask_remaining, 'sqft_living']
    / df.loc[mask_remaining, 'grade'].map(
        grade_ratio_orig
    )
).round().clip(lower=1)

# ==========================================
# Compare distributions
# ==========================================
dist_observed = (
    clean_df['bedrooms']
    .dropna()
    .value_counts(normalize=True)
    .sort_index()
)

dist_global = (
    bed_global_candidate
    .value_counts(normalize=True)
    .sort_index()
)

dist_grade = (
    bed_grade_candidate
    .value_counts(normalize=True)
    .sort_index()
)

comparison_remaining = (
    pd.concat(
        [
            dist_observed,
            dist_global,
            dist_grade,
        ],
        axis=1,
    )
    .fillna(0)
)

comparison_remaining.columns = [
    'Observed',
    'Global Final',
    'Grade Final',
]

print("Final bedroom imputation comparison:")

display(comparison_remaining)

Final bedroom imputation comparison:


,Observed,Global Final,Grade Final
bedrooms,,,
0.0,0.000537,0.0,0.0
1.0,0.009229,0.009207,0.009069
2.0,0.127789,0.130847,0.130893
3.0,0.453440,0.449591,0.449961
4.0,0.318326,0.31643,0.316384
5.0,0.075150,0.077592,0.0775
6.0,0.012842,0.013372,0.013233
7.0,0.001660,0.001897,0.001897
8.0,0.000586,0.000694,0.000694


### Dataset Completion Check

* Staged imputation outcomes verified to confirm absence of residual
  missingness in core structural housing variables  

* Dataset screened for remaining structurally implausible values
  following harmonization and reconstruction procedures  

* Empirical frequency distributions re-evaluated to ensure
  distributional integrity was preserved  

* Finalized dataset established as the analytical baseline for
  subsequent outlier diagnostics and statistical modeling  

In [604]:
print(
    "Applying final global fallback imputation for any "
    "remaining missing bedroom values:"
)

# Build sqft-per-bedroom ratio from ORIGINAL data
df_temp_orig = clean_df.loc[
    clean_df['bedrooms'].notna()
].copy()

df_temp_orig['sqft_per_bedroom'] = (
    df_temp_orig['sqft_living']
    / df_temp_orig['bedrooms']
)

global_ratio_final = df_temp_orig[
    'sqft_per_bedroom'
].median()

mask_remaining = df['bedrooms'].isna()

print(
    f"Remaining missing bedrooms before final fill: "
    f"{mask_remaining.sum()}"
)

# Final fill
df.loc[mask_remaining, 'bedrooms'] = (
    df.loc[mask_remaining, 'sqft_living']
    / global_ratio_final
).round().clip(lower=1)

print(
    f"Remaining missing bedrooms after final fill: "
    f"{df['bedrooms'].isna().sum()}"
)

print("\nFinal missing value counts across dataset:")

display(
    df
    .isnull()
    .sum()
)

Applying final global fallback imputation for any remaining missing bedroom values:
Remaining missing bedrooms before final fill: 61
Remaining missing bedrooms after final fill: 0

Final missing value counts across dataset:


id               0
date             0
price            0
bedrooms         0
bathrooms        0
sqft_living      0
sqft_lot         0
floors           0
waterfront       0
view             0
condition        0
grade            0
sqft_above       0
sqft_basement    0
yr_built         0
yr_renovated     0
zipcode          0
lat              0
long             0
sqft_living15    0
sqft_lot15       0
bed_missing      0
bath_missing     0
lot_missing      0
dtype: int64

### Export Final Clean Dataset

* Cleaned dataset exported following completion of structural
  harmonization and staged imputation procedures  

* Export reflects correction of deterministic data entry errors and
  preservation of empirical distributional structure  

* Measurement resolution maintained consistently across reconstructed
  housing attributes  

* Resulting dataset serves as the finalized input for modeling and
  collaborative review  

In [605]:
# Export finalized clean dataset
final_df = df.copy()

final_df.to_csv(
    "houses_clean_preprocessed.csv",
    index=False,
)

print("Final cleaned dataset exported.")

Final cleaned dataset exported.
